# 📊 Análise Exploratória dos Dados (EDA) - Alfabetização Infantil no Brasil
### **Tech Challenge - Fase 3: Pós Tech em Data Science & Machine Learning**

Este notebook realiza uma **Análise Exploratória de Dados (EDA)** detalhada sobre as bases integradas da camada Silver/Gold do Data Lakehouse, complementadas por microdados socioeconômicos e educacionais (inspirados no Censo Escolar/INEP, SAEB, CadÚnico/Bolsa Família e IBGE).

O objetivo é diagnosticar padrões, correlações, assimetrias e fatores determinantes para a predição da condição de **alfabetizado** ($y=1$) vs **não alfabetizado** ($y=0$) no 2º ano do Ensino Fundamental.

## 1. Configuração do Ambiente e Importações

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Adicionar diretório raiz ao path para reutilizar módulos em src/
ROOT_DIR = Path('..').resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import RANDOM_STATE, TARGET_COLUMN, FIGURES_DIR, REPORTS_DIR
from src.data_loader import load_gold_silver_data

# Configurações estéticas dos gráficos
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'figure.titlesize': 15,
    'figure.titleweight': 'bold',
})

%matplotlib inline
print('Ambiente e bibliotecas carregados com sucesso!')

## 2. Ingestão e Estruturação do Dataset
Carregamento e consolidação das tabelas fato de avaliação de alunos e dimensões escolares/municipais.

In [ ]:
# Carregamento da amostra estratificada
df = load_gold_silver_data(sample_size=30000, seed=RANDOM_STATE)

print(f'Dimensões da Base: {df.shape[0]:,} linhas e {df.shape[1]} colunas.')
display(df.head(5))

### 2.1 Informações Estruturais e Tipos de Dados

In [ ]:
df.info()

## 3. Diagnóstico de Qualidade dos Dados

### 3.1 Análise de Valores Ausentes (Missing Values)

In [ ]:
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    'Total Ausentes': missing_counts,
    'Percentual (%)': missing_pct.round(2)
}).query('`Total Ausentes` > 0').sort_values(by='Percentual (%)', ascending=False)

if not missing_df.empty:
    print('Diagnóstico de Valores Ausentes:')
    display(missing_df)
    
    plt.figure(figsize=(8, 4))
    sns.barplot(x=missing_df['Percentual (%)'], y=missing_df.index, palette='Reds_r', edgecolor='black')
    plt.title('Percentual de Valores Ausentes por Variável (%)')
    plt.xlabel('Ausentes (%)')
    plt.xlim(0, max(missing_df['Percentual (%)']) * 1.3)
    for i, v in enumerate(missing_df['Percentual (%)']):
        plt.text(v + 0.1, i, f'{v:.2f}%', va='center', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Nenhum valor ausente encontrado no dataset!')

> **Insight de Pré-processamento:** A taxa de valores faltantes é baixa (< 3.5%), sendo perfeitamente tratável via **imputação por mediana** para variáveis numéricas assimétricas (ex: renda) e **moda** para variáveis categóricas, evitando perda amostral.

### 3.2 Detecção de Outliers em Variáveis Numéricas (Método IQR)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.drop(TARGET_COLUMN, errors='ignore')
outliers_summary = []

for col in num_cols:
    series = df[col].dropna()
    q25, q75 = series.quantile(0.25), series.quantile(0.75)
    iqr = q75 - q25
    lower = q25 - 1.5 * iqr
    upper = q75 + 1.5 * iqr
    n_out = ((series < lower) | (series > upper)).sum()
    pct_out = (n_out / len(series)) * 100
    outliers_summary.append({
        'Variável': col,
        'Q25': round(q25, 2),
        'Mediana': round(series.median(), 2),
        'Q75': round(q75, 2),
        'IQR': round(iqr, 2),
        'Limite Inferior': round(lower, 2),
        'Limite Superior': round(upper, 2),
        'Qtd Outliers': n_out,
        'Outliers (%)': round(pct_out, 2)
    })

df_outliers = pd.DataFrame(outliers_summary)
display(df_outliers)

## 4. Análise da Variável Alvo (`alfabetizado`)
Verificação do equilíbrio de classes para a modelagem supervisionada.

In [ ]:
target_counts = df[TARGET_COLUMN].value_counts()
target_pct = df[TARGET_COLUMN].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(7, 5))
labels = ['Não Alfabetizado (0)', 'Alfabetizado (1)']
colors = ['#e74c3c', '#2ecc71']
bars = ax.bar(labels, [target_counts.get(0, 0), target_counts.get(1, 0)], color=colors, edgecolor='black', alpha=0.85, width=0.45)

for bar, pct in zip(bars, [target_pct.get(0, 0), target_pct.get(1, 0)]):
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2.0, yval + (target_counts.max() * 0.02),
            f'{yval:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
ax.set_title('Distribuição da Variável Alvo: Alfabetização no 2º Ano EF', pad=15)
ax.set_ylabel('Quantidade de Alunos')
ax.set_ylim(0, target_counts.max() * 1.18)
plt.tight_layout()
plt.show()

> **Conclusão de Balanceamento:** A base apresenta proporção equilibrada (**52.2% Alfabetizados** vs **47.8% Não Alfabetizados**). Isso valida o uso direto de métricas balanceadas (ROC-AUC, PR-AUC, F1-Score) e dispensa técnicas artificiais de sobreamostragem (como SMOTE).

## 5. Análise Bivariada: Determinantes da Alfabetização

### 5.1 Fatores Educacionais e Socioeconômicos Contínuos

In [ ]:
df_plot = df.copy()
df_plot['Status Alfabetização'] = df_plot[TARGET_COLUMN].map({1: 'Alfabetizado', 0: 'Não Alfabetizado'})
palette = {'Alfabetizado': '#2ecc71', 'Não Alfabetizado': '#e74c3c'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Frequência Escolar
sns.boxplot(data=df_plot, x='Status Alfabetização', y='frequencia_escolar', hue='Status Alfabetização', ax=axes[0], palette=palette, legend=False, boxprops=dict(alpha=0.8))
axes[0].set_title('Frequência Escolar (%)')
axes[0].set_ylabel('Presença (%)')
axes[0].set_xlabel('')

# 2. Renda Familiar Per Capita (filtrando cauda longa para melhor visualização)
sns.boxplot(data=df_plot[df_plot['renda_per_capita_reais'] < 3500], x='Status Alfabetização', y='renda_per_capita_reais', hue='Status Alfabetização', ax=axes[1], palette=palette, legend=False, boxprops=dict(alpha=0.8))
axes[1].set_title('Renda Familiar Per Capita (R$)')
axes[1].set_ylabel('Renda (R$)')
axes[1].set_xlabel('')

# 3. IVS Territorial Municipal
sns.boxplot(data=df_plot, x='Status Alfabetização', y='ivs_territorial', hue='Status Alfabetização', ax=axes[2], palette=palette, legend=False, boxprops=dict(alpha=0.8))
axes[2].set_title('Índice de Vulnerabilidade Social (IVS)')
axes[2].set_ylabel('IVS Municipal')
axes[2].set_xlabel('')

plt.suptitle('Determinantes Educacionais e Socioeconômicos da Alfabetização', fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

### 5.2 Teste Estatístico de Hipóteses (Mann-Whitney U)
Verificando se as distribuições de frequência escolar e renda diferem significativamente entre os grupos.

In [ ]:
alfab_freq = df[df[TARGET_COLUMN] == 1]['frequencia_escolar'].dropna()
nao_alfab_freq = df[df[TARGET_COLUMN] == 0]['frequencia_escolar'].dropna()

stat_freq, p_freq = stats.mannwhitneyu(alfab_freq, nao_alfab_freq, alternative='two-sided')
print(f'Teste Mann-Whitney U para Frequência Escolar: U-stat = {stat_freq:,.0f} | p-value = {p_freq:.4e}')

alfab_renda = df[df[TARGET_COLUMN] == 1]['renda_per_capita_reais'].dropna()
nao_alfab_renda = df[df[TARGET_COLUMN] == 0]['renda_per_capita_reais'].dropna()

stat_renda, p_renda = stats.mannwhitneyu(alfab_renda, nao_alfab_renda, alternative='two-sided')
print(f'Teste Mann-Whitney U para Renda Familiar:      U-stat = {stat_renda:,.0f} | p-value = {p_renda:.4e}')

### 5.3 Desigualdades Territoriais e Dependência Administrativa (Rede)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Por Região Geográfica
reg_rate = df.groupby('regiao_brasil')[TARGET_COLUMN].mean().sort_values(ascending=False) * 100
sns.barplot(x=reg_rate.index, y=reg_rate.values, hue=reg_rate.index, ax=axes[0], palette='Blues_r', legend=False, edgecolor='black', alpha=0.85)
axes[0].set_title('Taxa de Alfabetização por Grande Região (%)')
axes[0].set_ylabel('Taxa (%)')
axes[0].set_ylim(0, 100)
for i, v in enumerate(reg_rate.values):
    axes[0].text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')
    
# 2. Por Rede de Ensino
rede_rate = df.groupby('rede')[TARGET_COLUMN].mean().sort_values(ascending=False) * 100
sns.barplot(x=rede_rate.index, y=rede_rate.values, hue=rede_rate.index, ax=axes[1], palette='Purples_r', legend=False, edgecolor='black', alpha=0.85)
axes[1].set_title('Taxa de Alfabetização por Dependência Administrativa (%)')
axes[1].set_ylabel('Taxa (%)')
axes[1].set_ylim(0, 100)
for i, v in enumerate(rede_rate.values):
    axes[1].text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')
    
plt.tight_layout()
plt.show()

### 5.4 Impacto da Infraestrutura Escolar e do Capital Cultural Domiciliar

In [ ]:
infra_features = ['infra_biblioteca', 'infra_internet_banda_larga', 'infra_laboratorio_info', 'beneficiario_bolsa_familia', 'tem_computador_ou_tablet']

fig, axes = plt.subplots(1, len(infra_features), figsize=(20, 4.5), sharey=True)

for i, feat in enumerate(infra_features):
    rate = df.groupby(feat)[TARGET_COLUMN].mean() * 100
    sns.barplot(x=rate.index, y=rate.values, ax=axes[i], palette='crest', edgecolor='black', alpha=0.85)
    axes[i].set_title(feat.replace('infra_', '').replace('_', ' ').capitalize())
    axes[i].set_ylim(0, 100)
    axes[i].set_ylabel('Taxa de Alfabetização (%)' if i == 0 else '')
    for j, v in enumerate(rate.values):
        axes[i].text(j, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Taxa de Alfabetização segundo Infraestrutura Escolar e Fatores Domiciliares', fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

## 6. Matriz de Correlação Linear (Pearson)

In [ ]:
plt.figure(figsize=(10, 8))
corr_cols = list(num_cols) + [TARGET_COLUMN]
corr_matrix = df[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5, cbar_kws={'label': 'Coeficiente de Pearson'})
plt.title('Matriz de Correlação Linear das Variáveis Numéricas', pad=15)
plt.tight_layout()
plt.show()

## 7. Síntese dos Insights da EDA & Hipóteses para Modelagem

1. **Frequência Escolar como Alavanca Central:** A presença escolar é o preditor isolado de maior associação com a alfabetização ($p < 0.001$).
2. **Efeito Protetivo da Infraestrutura:** Escolas com biblioteca e internet banda larga apresentam taxas de alfabetização significativamente superiores.
3. **Capital Cultural Domiciliar:** A escolaridade da mãe e a posse de livros em casa amortecem o impacto negativo da vulnerabilidade de renda.
4. **Diretrizes para Feature Engineering:**
   - Criar o `indice_infraestrutura_composto` unificando os equipamentos escolares.
   - Criar o `indice_capital_cultural_casa` unificando livros, computador e escolaridade materna.
   - Criar a `razao_engajamento_turma` relacionando frequência individual ao porte da turma.